# Continuous-Time Generative Models: CNF (MLE) vs Flow Matching

**Goal:** Train the *same* neural velocity-field architecture $f_\theta(t,z)$ in two different ways:

1. **CNF with maximum likelihood:** solve an augmented ODE **backward** through each real data point to evaluate $\log p_1^\theta(x)$, then minimize negative log-likelihood.
2. **Flow Matching (FM):** construct interpolated points $z_t=(1-t)z_0+tz_1$ and directly regress the target velocities $z_1-z_0$ **without solving an ODE during training**.

At generation time, **both** models solve the same kind of forward Neural ODE:

$$\boxed{\frac{dz(t)}{dt}=f_\theta(t,z(t)),\qquad z(0)\sim p_0=\mathcal N(0,I_2).}$$

We will visualize intermediate distributions, vector fields, training curves, and held-out negative log-likelihood. Everything is 2D so the divergence can be computed exactly.


## 0. Setup

Run this notebook in Colab or a Python environment with PyTorch. The first cell installs `torchdiffeq` if necessary. `FAST_MODE=True` keeps a classroom demo manageable; switch it off for longer training.

In [ ]:
# Colab / local setup: install only if missing.
import importlib.util, subprocess, sys
if importlib.util.find_spec("torchdiffeq") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torchdiffeq"])

import time, copy, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchdiffeq import odeint

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
FAST_MODE = True       # False: longer training and generally better fit
BATCH_SIZE = 192 if FAST_MODE else 256
MLE_STEPS = 150 if FAST_MODE else 500
FM_STEPS = 750 if FAST_MODE else 2200
LR_MLE, LR_FM = 1e-3, 1e-3
N_ODE_STEPS_TRAIN = 8 if FAST_MODE else 12
N_ODE_STEPS_SAMPLE = 48  # more accurate sampling/visualization
print("Device:", DEVICE, "| train iterations: MLE", MLE_STEPS, "/ FM", FM_STEPS)


## 1. A spread-out, multimodal target distribution

We use a **known mixture of eight 2D Gaussians** with non-uniform radii, different covariance orientations, and well-separated modes. Knowing its density lets us compute additional evaluation metrics, but **training only uses samples**, just as with an ordinary dataset.

**Important:** the underlying Gaussian *base* $p_0$ is different from this Gaussian *mixture* $p_{\mathrm{data}}$.

In [ ]:
NUM_MODES = 8
angles = np.linspace(0, 2*np.pi, NUM_MODES, endpoint=False) + .13
radii = np.array([4.8, 4.3, 5.4, 4.6, 5.1, 4.2, 5.5, 4.5])
means_np = np.stack([radii*np.cos(angles), 0.85*radii*np.sin(angles)], axis=1)
means = torch.tensor(means_np, dtype=DTYPE, device=DEVICE)
scales_np = np.array([[.32,.55],[.48,.33],[.33,.44],[.40,.35],
                      [.46,.33],[.34,.55],[.39,.36],[.35,.50]])
scales = torch.tensor(scales_np, dtype=DTYPE, device=DEVICE)
rot_angles = torch.tensor(angles + .65, dtype=DTYPE, device=DEVICE)
rots = torch.stack([
    torch.stack([torch.cos(rot_angles), -torch.sin(rot_angles)], dim=-1),
    torch.stack([torch.sin(rot_angles),  torch.cos(rot_angles)], dim=-1)
], dim=-2)  # [K,2,2]

@torch.no_grad()
def sample_target(n):
    k = torch.randint(0, NUM_MODES, (n,), device=DEVICE)
    eps = torch.randn(n,2,device=DEVICE)
    return means[k] + torch.bmm(rots[k], (scales[k]*eps).unsqueeze(-1)).squeeze(-1)

@torch.no_grad()
def sample_base(n):
    return torch.randn(n,2,device=DEVICE)

def base_log_prob(z):
    return -0.5*(z**2).sum(dim=-1) - np.log(2*np.pi)  # d=2

def target_log_prob(z):
    # Density is used for evaluation/visualization ONLY, not either training loss.
    d = z[:,None,:] - means[None,:,:]        # [B,K,2]
    local = torch.einsum('kij,bkj->bki', rots.transpose(-1,-2), d)
    sq = ((local / scales[None,:,:])**2).sum(-1)
    logdet = scales.log().sum(-1)
    log_components = -0.5*sq - logdet[None,:] - np.log(2*np.pi)
    return torch.logsumexp(log_components - np.log(NUM_MODES), dim=1)

fig, ax = plt.subplots(1,2,figsize=(11,4.5),sharex=True,sharey=True)
z0 = sample_base(2500).cpu().numpy()
z1 = sample_target(2500).cpu().numpy()
ax[0].scatter(z0[:,0],z0[:,1],s=4,alpha=.4)
ax[0].set_title('Base distribution $p_0$ (standard Gaussian)')
ax[1].scatter(z1[:,0],z1[:,1],s=4,alpha=.5)
ax[1].set_title('Target distribution $p_{data}$ (8 modes)')
for a in ax:
    a.set(xlim=(-7,7),ylim=(-7,7),xlabel='$z_1$',ylabel='$z_2$')
    a.set_aspect('equal'); a.grid(alpha=.15)
plt.tight_layout(); plt.show()


## 2. Shared Neural ODE vector field

Both models use the **same architecture** $f_\theta:[0,1]\times\mathbb R^2\to\mathbb R^2$ and the same initial parameters. We choose smooth `Tanh` activations, useful for differentiating the vector field with respect to its spatial input.

The network is evaluated at different $(t,z)$ pairs. Its output is a **velocity vector**, not a position or a probability.

In [ ]:
class VectorField(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 2)
        )
        # Small initial velocities make backward training more stable.
        nn.init.normal_(self.net[-1].weight, std=0.01)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, t, z):
        # t is a scalar tensor; z shape [B,2]
        tt = torch.ones_like(z[:, :1]) * t
        return self.net(torch.cat([z, tt], dim=-1))

initial = VectorField().to(DEVICE)
model_mle = copy.deepcopy(initial)
model_fm = copy.deepcopy(initial)
print("Trainable parameters:", sum(p.numel() for p in initial.parameters()))


## 3. CNF trained using Maximum Likelihood

The instantaneous change-of-variables equation is

$$\frac{d}{dt}\log p_t(z(t)) = -\nabla_z\cdot f_\theta(t,z(t))
=-\mathrm{Tr}\left(\frac{\partial f_\theta}{\partial z}(t,z(t))\right).$$

For each observed $x=z(1)$, integrate **backward** the augmented state $(z,a)$ from $t=1$ to $t=0$:

$$\frac{dz}{dt}=f_\theta(t,z),\qquad
\frac{da}{dt}=-\nabla_z\cdot f_\theta(t,z),\qquad
z(1)=x,\;a(1)=0.$$

We get $z(0)$ and an *accumulated density correction* $a(0)$. The arbitrary choice $a(1)=0$ **does not** say $\log p_1(x)=0$; the actual log-density at $t=1$ is unknown before solving.

$$\boxed{\log p_1^\theta(x)=\log p_0(z(0))-a(0).}$$

**Why calculate divergence exactly?** In 2D it is just $\partial f_{\theta,1}/\partial z_1+\partial f_{\theta,2}/\partial z_2$. In high dimensions one could use stochastic trace estimators (e.g. Hutchinson).

**Important:** the inverse-mapped data $z(0)$ are *not automatically Gaussian*. They approach $p_0$ in distribution only as the learned flow approaches a suitable fit.

In [ ]:
def exact_divergence(f, t, z):
    """tr(df/dz), per sample. Create graph for gradients w.r.t. theta."""
    with torch.enable_grad():
        z = z.requires_grad_(True)
        velocity = f(t,z)
        trace = torch.zeros(z.shape[0],device=z.device,dtype=z.dtype)
        for j in range(z.shape[1]):
            # One VJP per component; we need only the diagonal entries.
            grad_j = torch.autograd.grad(
                velocity[:,j].sum(), z, create_graph=True, retain_graph=True
            )[0][:,j]
            trace = trace + grad_j
    return trace

class AugmentedCNF(nn.Module):
    def __init__(self, velocity_field):
        super().__init__()
        self.f = velocity_field
    def forward(self, t, state):
        z, a = state                      # z:[B,2], a:[B,1]
        with torch.enable_grad():
            z = z.requires_grad_(True)
            dz = self.f(t,z)
            div = exact_divergence(self.f,t,z)
        da = -div.unsqueeze(-1)
        return dz, da

def cnf_log_prob(model, x, n_steps=N_ODE_STEPS_TRAIN):
    a1 = torch.zeros(x.shape[0],1,device=x.device,dtype=x.dtype)
    # backward from 1 -> 0; fixed-step RK4 for stable, reproducible teaching.
    z_series, a_series = odeint(
        AugmentedCNF(model), (x,a1),
        torch.tensor([1.,0.],device=x.device),
        method='rk4', options={'step_size':1./n_steps}
    )
    z0, a0 = z_series[-1], a_series[-1].squeeze(-1)
    log_px = base_log_prob(z0) - a0
    return log_px, z0, a0

# Check the sign on a known field f(z) = c*z: divergence = 2c.
class LinearExpansion(nn.Module):
    def __init__(self, c=.3):
        super().__init__()
        self.c=nn.Parameter(torch.tensor(float(c)))
    def forward(self,t,z): return self.c*z

test_field = LinearExpansion().to(DEVICE)
x_test = torch.tensor([[1.,-2.]],device=DEVICE)
lp,z_test,a_test = cnf_log_prob(test_field,x_test,n_steps=16)
expected = base_log_prob(x_test*torch.exp(-test_field.c)) - 2*test_field.c
print('Numerical log p1:', lp.item(), '| analytic:', expected.item())
assert torch.allclose(lp, expected,atol=3e-4), 'Check the divergence sign.'


### MLE training loop

Each iteration: **real batch → backward augmented solve → compute Gaussian base log-density + correction → minimize NLL → backpropagate through solver → Adam step**.

The augmented ODE is much more expensive than a network forward pass because divergence requires spatial derivatives at *every solver evaluation*.

In [ ]:
optimizer_mle = torch.optim.Adam(model_mle.parameters(),lr=LR_MLE)
mle_losses = []
start = time.perf_counter()
for step in range(MLE_STEPS):
    model_mle.train()
    x = sample_target(BATCH_SIZE)
    log_px, z_back, correction = cnf_log_prob(model_mle,x)
    loss = -log_px.mean()
    optimizer_mle.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_mle.parameters(),5.0)
    optimizer_mle.step()
    mle_losses.append(float(loss.detach().cpu()))
    if (step+1) % max(1,MLE_STEPS//5)==0:
        print(f'MLE step {step+1:4d}/{MLE_STEPS}: NLL {mle_losses[-1]:.3f}')
mle_seconds = time.perf_counter()-start
print(f'MLE training elapsed: {mle_seconds:.1f} seconds')


## 4. Flow Matching (conditional straight-line paths)

Draw independent samples $z_0\sim p_0$ and $z_1\sim p_{\mathrm{data}}$, then $t\sim\mathcal U(0,1)$. Construct

$$\boxed{z_t=(1-t)z_0+tz_1},\qquad \boxed{u_t=z_1-z_0}.$$

This $u_t$ is the **known velocity of the chosen conditional interpolation**, not necessarily the velocity of the final *marginal probability flow*. We regress

$$\boxed{\mathcal J_{\mathrm{FM}}(\theta)=\mathbb E_{t,z_0,z_1}
\left[\|f_\theta(t,z_t)-(z_1-z_0)\|_2^2\right].}$$

Under squared error, the optimal field is the **conditional mean** $\mathbb E[z_1-z_0\mid z_t=z]$: different pairs may pass through the same intermediate location with different velocities. Thus, **the learned ODE trajectories need not be straight**, even though the *training paths* are straight.

No ODE solver and no divergence are needed during FM training.

In [ ]:
# Visualize a few paired interpolations used only as supervised training examples.
start_points = sample_base(7).cpu().numpy()
end_points = sample_target(7).cpu().numpy()
fig,ax=plt.subplots(figsize=(7,6))
cloud0=sample_base(700).cpu().numpy(); cloud1=sample_target(700).cpu().numpy()
ax.scatter(cloud0[:,0],cloud0[:,1],s=5,alpha=.12,label='Base samples')
ax.scatter(cloud1[:,0],cloud1[:,1],s=5,alpha=.15,label='Data samples')
for a,b in zip(start_points,end_points):
    ax.plot([a[0],b[0]],[a[1],b[1]],lw=1.5,alpha=.65)
    mid=.5*(a+b)
    ax.scatter([mid[0]],[mid[1]],s=42,marker='x')
ax.set(title='Random pairs and intermediate points ($t=0.5$)',xlabel='$z_1$',ylabel='$z_2$',xlim=(-7,7),ylim=(-7,7))
ax.set_aspect('equal'); ax.legend(); plt.show()


In [ ]:
optimizer_fm = torch.optim.Adam(model_fm.parameters(),lr=LR_FM)
fm_losses=[]
start=time.perf_counter()
for step in range(FM_STEPS):
    model_fm.train()
    z0 = sample_base(BATCH_SIZE)
    z1 = sample_target(BATCH_SIZE)
    t = torch.rand(BATCH_SIZE,1,device=DEVICE)
    zt = (1-t)*z0 + t*z1
    target_velocity = z1-z0
    # model accepts a scalar time; batching with distinct t needs a slight adapter:
    # concatenate per-sample times directly to match the same underlying MLP.
    pred = model_fm.net(torch.cat([zt,t],dim=1))
    loss = (pred-target_velocity).square().sum(dim=1).mean()
    optimizer_fm.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_fm.parameters(),5.)
    optimizer_fm.step()
    fm_losses.append(float(loss.detach().cpu()))
    if (step+1)%max(1,FM_STEPS//5)==0:
        print(f'FM step {step+1:4d}/{FM_STEPS}: velocity MSE {fm_losses[-1]:.3f}')
fm_seconds=time.perf_counter()-start
print(f'FM training elapsed: {fm_seconds:.1f} seconds')


## 5. Generate samples: same forward Neural ODE, different learned fields

For **both** models, draw $z(0)\sim\mathcal N(0,I)$ and solve

$$\frac{dz(t)}{dt}=f_\theta(t,z(t))\qquad \text{from }t=0\text{ to }1.$$

**Sampling uses an ODE solver for both methods.** Here we use RK4 with finer steps than in training to improve visualization accuracy.

In [ ]:
@torch.no_grad()
def generate(model, n=1800, steps=N_ODE_STEPS_SAMPLE, return_path=False):
    model.eval()
    initial = sample_base(n)
    times = torch.linspace(0,1,9,device=DEVICE) if return_path else torch.tensor([0.,1.],device=DEVICE)
    trajectory = odeint(model, initial, times, method='rk4', options={'step_size':1./steps})
    return trajectory.detach().cpu().numpy() if return_path else trajectory[-1].detach().cpu().numpy()

real = sample_target(2000).cpu().numpy()
base = sample_base(2000).cpu().numpy()
mle_samples = generate(model_mle,n=2000)
fm_samples = generate(model_fm,n=2000)
fig,ax=plt.subplots(1,4,figsize=(16,4),sharex=True,sharey=True)
for a,cloud,title in zip(ax,[base,real,mle_samples,fm_samples],
                         ['Base Gaussian','Real target','CNF trained by MLE','Flow Matching']):
    a.scatter(cloud[:,0],cloud[:,1],s=4,alpha=.52)
    a.set(title=title,xlim=(-7,7),ylim=(-7,7),xlabel='$z_1$')
    a.set_aspect('equal');a.grid(alpha=.1)
ax[0].set_ylabel('$z_2$');plt.tight_layout();plt.show()


### How does the generated distribution evolve over time?

These are actual **learned ODE trajectories**, not the artificial straight interpolation segments used to train FM.

In [ ]:
fig,axs=plt.subplots(2,5,figsize=(16,6),sharex=True,sharey=True)
times=np.linspace(0,1,9)
for row,(name,model) in enumerate([('MLE',model_mle),('FM',model_fm)]):
    path=generate(model,n=1300,return_path=True)
    for col,idx in enumerate([0,2,4,6,8]):
        axs[row,col].scatter(path[idx,:,0],path[idx,:,1],s=3,alpha=.45)
        axs[row,col].set(title=f'{name}: t={times[idx]:.2f}',xlim=(-7,7),ylim=(-7,7))
        axs[row,col].set_aspect('equal');axs[row,col].grid(alpha=.1)
plt.tight_layout();plt.show()


## 6. Comparison: losses, runtime and held-out likelihood

**Do not directly compare the numerical values of the training losses:** MLE uses negative log-density, FM uses velocity squared error (different units/objectives).

Runtime is measured for the **configured total number of training steps**. Because FM and MLE are trained for different numbers of steps, compare also *seconds per training step*; this is only a hardware-dependent illustrative benchmark, not a controlled accuracy-at-equal-compute experiment.

**Optional quantitative check:** even though FM does *not* optimize likelihood, once its vector field is learned it still defines a CNF. We can compute its likelihood via the same augmented ODE. This is a key theoretical connection.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,3.5))
ax[0].plot(mle_losses);ax[0].set(title='MLE: negative log-likelihood',xlabel='Optimizer step',ylabel='NLL')
ax[1].plot(fm_losses);ax[1].set(title='FM: velocity regression loss',xlabel='Optimizer step',ylabel='Squared error')
for a in ax:a.grid(alpha=.2)
plt.tight_layout();plt.show()
print(f'MLE: {mle_seconds:.1f}s total / {1000*mle_seconds/MLE_STEPS:.1f} ms per step')
print(f'FM:  {fm_seconds:.1f}s total / {1000*fm_seconds/FM_STEPS:.1f} ms per step')


In [ ]:
# Optional but instructive: score both learned CNFs on identical held-out data.
# 2D: exact divergence for likelihood of either trained field.
EVAL_N = 192 if FAST_MODE else 384
heldout = sample_target(EVAL_N)

def heldout_nll(model, x, batch=48, steps=24):
    vals=[]
    model.eval()
    for chunk in x.split(batch):
        # Avoid torch.no_grad(): spatial autograd is essential for divergence.
        lp,_,_ = cnf_log_prob(model,chunk,n_steps=steps)
        vals.append((-lp).detach())
    return torch.cat(vals).mean().item()

nll_mle = heldout_nll(model_mle,heldout)
nll_fm = heldout_nll(model_fm,heldout)
true_entropy_estimate = (-target_log_prob(heldout)).mean().item()
print(f'Ground-truth entropy estimate on held-out samples: {true_entropy_estimate:.3f}')
print(f'MLE learned model — held-out NLL:            {nll_mle:.3f}')
print(f'FM learned model — held-out NLL:             {nll_fm:.3f}')
print('Lower NLL is better; estimated true entropy is a statistical reference, not a hard per-batch lower bound.')


## 7. Inspect the learned vector fields

These plots show $f_\theta(t,z)$ evaluated over space for several fixed values of $t$. The arrows represent **velocities**, not trajectories; trajectories are obtained by integrating them.

In [ ]:
@torch.no_grad()
def draw_field(ax, model, t, n=17):
    xx,yy=np.meshgrid(np.linspace(-6.7,6.7,n),np.linspace(-6.7,6.7,n))
    z=torch.tensor(np.stack([xx.ravel(),yy.ravel()],axis=-1),dtype=DTYPE,device=DEVICE)
    vv=model(torch.tensor(float(t),device=DEVICE),z).cpu().numpy()
    u,v=vv[:,0].reshape(xx.shape),vv[:,1].reshape(yy.shape)
    ax.quiver(xx,yy,u,v,angles='xy',scale_units='xy',scale=8,width=.003)
    ax.set(xlim=(-7,7),ylim=(-7,7),xlabel='$z_1$',ylabel='$z_2$')
    ax.set_aspect('equal')
fig,axes=plt.subplots(2,3,figsize=(12,8),sharex=True,sharey=True)
for row,(name,m) in enumerate([('MLE',model_mle),('FM',model_fm)]):
    for col,t in enumerate([0.,.5,1.]):
        draw_field(axes[row,col],m,t)
        axes[row,col].set_title(f'{name} field at t={t:g}')
plt.tight_layout();plt.show()


## 8. Key takeaways and exercises

| | CNF + Maximum Likelihood | Flow Matching |
|---|---|---|
| Training data | Real samples $x$ | Pairs $(z_0,z_1)$ and artificial $z_t$ |
| Target | Maximize $\log p_1^\theta(x)$ | Regress conditional velocity $z_1-z_0$ |
| ODE solver during training | **Yes** (backward augmented ODE) | **No** |
| Divergence during training | **Yes** | **No** |
| ODE solver during generation | **Yes** | **Yes** |
| Likelihood evaluation after training | Via augmented ODE | Also possible via augmented ODE |

**Exercises**

1. Set `FAST_MODE=False`. Compare quality at longer training budgets.
2. Increase the number of Gaussian modes, or spread them farther apart. Which method is more sensitive?
3. Replace independent $(z_0,z_1)$ pairs with a structured coupling. What changes in the conditional target velocities?
4. In MLE, vary the RK4 step size. Does the NLL computed at 8 steps agree with the finer 24-step evaluation?
5. Measure both models' **held-out NLL** with the *same* numerical solver settings (implemented above). Does the FM model have a valid likelihood even though it was not trained by MLE?

**Numerical caveat:** exact invertibility and the density identity refer to the continuous flow. Numerical ODE solvers introduce discretization error, and imperfect training may leave significant discrepancies between the learned model and the target distribution.
